# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [ ]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [ ]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [ ]:
import re

def agent(query: str):
    """Route a query to the correct tool and return a structured JSON-style dict."""
    try:
        if not isinstance(query, str) or not query.strip():
            return {"type": "error", "result": "Empty or invalid query"}

        query_lower = query.lower()

        # --- Route 1: Calculation ---
        if "calculate" in query_lower:
            match = re.search(r"[-+*/().\d\s]*\d[-+*/().\d\s]*", query)
            if not match:
                return {"type": "error", "result": "No valid expression found to calculate"}

            expression = match.group().strip()

            if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
                return {"type": "error", "result": "Expression contains invalid characters"}

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                return {"type": "error", "result": calc_result}

            return {"type": "calculation", "result": calc_result}

        # --- Route 2: Keyword extraction ---
        elif "keywords" in query_lower:
            text = re.sub(r"(?i)^.*?keywords( from)?\s*", "", query).strip()
            if not text:
                text = query

            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # --- Route 3: General fallback ---
        else:
            return {"type": "general", "result": f"I received your query: '{query}'. This looks like a general question, not a calculation or keyword request."}

    except Exception as e:
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

## 🚀 Bonus: Improved Routing, Logging & Extra Tools

This section extends the base agent with three bonus features:
- **Improved routing** — uses whole-word matching so substrings like "recalculate" don't misfire, and resolves cases where a query matches more than one route.
- **Logging** — every call is recorded (query, chosen route, result) using Python's `logging` module.
- **More tools** — a word/character counter and a simple km↔miles unit converter.

Running the cells below redefines `agent()`, so everything after this point (including the original test cases and interactive loop) will use the upgraded version.

In [ ]:
def convert_units(text: str) -> str:
    """Convert a number of km to miles, or miles to km, based on the text."""
    try:
        match = re.search(r"[-+]?\d*\.?\d+", text)
        if not match:
            return "No numeric value found to convert"
        value = float(match.group())

        text_lower = text.lower()
        km_idx = text_lower.find("km")
        mile_idx = text_lower.find("mile")

        if km_idx == -1 and mile_idx == -1:
            return "No unit (km/miles) found to convert"

        if mile_idx != -1 and (km_idx == -1 or mile_idx < km_idx):
            source = "miles"
        else:
            source = "km"

        if source == "miles":
            result = value * 1.60934
            return f"{value} miles = {result:.2f} km"
        else:
            result = value / 1.60934
            return f"{value} km = {result:.2f} miles"
    except Exception:
        return "Error in conversion"

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("agent_logger")

call_log = []

def log_call(query: str, route: str, result):
    entry = {"query": query, "route": route, "result": result}
    call_log.append(entry)
    logger.info(f"query={query!r} | route={route} | result={result}")

In [ ]:
def agent(query: str):
    """Route a query to the correct tool and return a structured JSON-style dict.
    Bonus version: whole-word route matching, priority handling for overlapping
    keywords, logging of every call, and two extra tools (word count, unit conversion).
    """
    try:
        if not isinstance(query, str) or not query.strip():
            result = {"type": "error", "result": "Empty or invalid query"}
            log_call(query, "error", result["result"])
            return result

        query_lower = query.lower()

        wants_calc = bool(re.search(r"\bcalculate\b", query_lower))
        wants_keywords = bool(re.search(r"\bkeywords\b", query_lower))
        wants_count = bool(re.search(r"\bcount\b", query_lower))
        wants_convert = bool(re.search(r"\bconvert\b", query_lower)) and ("km" in query_lower or "mile" in query_lower)

        # Priority order: calculate > convert > count > keywords > general
        if wants_calc:
            match = re.search(r"[-+*/().\d\s]*\d[-+*/().\d\s]*", query)
            if not match:
                result = {"type": "error", "result": "No valid expression found to calculate"}
                log_call(query, "calculation", result["result"])
                return result

            expression = match.group().strip()
            if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
                result = {"type": "error", "result": "Expression contains invalid characters"}
                log_call(query, "calculation", result["result"])
                return result

            calc_result = calculator(expression)
            if calc_result == "Error in calculation":
                result = {"type": "error", "result": calc_result}
            else:
                result = {"type": "calculation", "result": calc_result}
            log_call(query, "calculation", result["result"])
            return result

        elif wants_convert:
            conv_result = convert_units(query)
            result = {"type": "conversion", "result": conv_result}
            log_call(query, "conversion", conv_result)
            return result

        elif wants_count:
            text = re.sub(r"(?i)^.*?count( the)?( words?| characters?)?( in| of)?\s*", "", query).strip()
            if not text:
                text = query
            count_result = count_words(text)
            result = {"type": "count", "result": count_result}
            log_call(query, "count", count_result)
            return result

        elif wants_keywords:
            text = re.sub(r"(?i)^.*?keywords( from)?\s*", "", query).strip()
            if not text:
                text = query
            keywords = extract_keywords(text)
            result = {"type": "keywords", "result": keywords}
            log_call(query, "keywords", keywords)
            return result

        else:
            general_result = f"I received your query: '{query}'. This looks like a general question, not a calculation or keyword request."
            result = {"type": "general", "result": general_result}
            log_call(query, "general", general_result)
            return result

    except Exception as e:
        result = {"type": "error", "result": f"Unexpected error: {str(e)}"}
        log_call(query, "error", result["result"])
        return result

In [ ]:
bonus_queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Count the words in this sentence please",
    "Convert 10 km to miles",
    "Please recalculate my budget"
]

for q in bonus_queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

print("\nCall log so far:")
for entry in call_log:
    print(entry)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['industries', 'transforming', 'intelligence', 'artificial']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. This looks like a general question, not a calculation or keyword request."}
--------------------------------------------------
Query: Count the words in this sentence please
Response: {'type': 'count', 'result': {'word_count': 3, 'char_count': 20}}
--------------------------------------------------
Query: Convert 10 km to miles
Response: {'type': 'conversion', 'result': '10.0 km = 6.21 miles'}
--------------------------------------------------
Query: Please recalculate my budget
Response: {'type': 'general', '

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): count word in sentence i am here
Response: {'type': 'count', 'result': {'word_count': 4, 'char_count': 18}}
Enter query (type 'exit' to stop): count word in sentence i am not here
Response: {'type': 'count', 'result': {'word_count': 5, 'char_count': 22}}
Enter query (type 'exit' to stop): count the words
Response: {'type': 'count', 'result': {'word_count': 3, 'char_count': 15}}
Enter query (type 'exit' to stop): count me
Response: {'type': 'count', 'result': {'word_count': 1, 'char_count': 2}}
Enter query (type 'exit' to stop): count the sentences
Response: {'type': 'count', 'result': {'word_count': 1, 'char_count': 9}}
Enter query (type 'exit' to stop): exit
